[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1kCktQ6oXUZ0CDryXBjja4RyRfQuxFBwG/view?usp=drive_link)

# Agent Evaluation – Pre-Captured Traces

This notebook demonstrates how to evaluate agents when you already have conversation logs (traces). No agent runs during evaluation — Floeval scores the traces directly.

**Objectives**
- Install Floeval and configure credentials
- Build an in-memory `AgentDataset` with full traces
- Configure `AgentEvaluation` with LLM-judge metrics
- Run the evaluation and inspect the summary

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 1. Imports

The following cell imports the agent evaluation components, the dataset schemas, and the message types. Traces use `messages` with roles `human`, `ai`, and `tool`.

In [ ]:
from floeval.api.agent_evaluation import AgentEvaluation
from floeval.config.schemas.io.agent_dataset import (
    AgentDataset,
    AgentSample,
    AgentTrace,
    AIMessage,
    HumanMessage,
    ToolMessage,
    ToolCall,
)
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 2. Build a Full Agent Sample

An `AgentSample` is created with a complete trace. The trace includes a human message, AI turn (with optional tool_calls), tool result, and final AI response.

In [ ]:
sample = AgentSample(
    user_input="Find weather for Paris and summarize.",
    reference_outcome="A concise weather summary for Paris.",
    trace=AgentTrace(
        messages=[
            HumanMessage(content="Find weather for Paris and summarize."),
            AIMessage(content="I will check weather first."),
            ToolMessage(content="Paris: 18C, light rain", tool_name="weather_lookup"),
            AIMessage(content="Paris is 18C with light rain today."),
        ],
        final_response="Paris is 18C with light rain today.",
    ),
)

dataset = AgentDataset(samples=[sample])
print(f"Dataset loaded: {len(dataset.samples)} full sample(s)")

## 3. Configure the LLM

The LLM configuration is built for agent metrics such as `goal_achievement` that use an LLM judge. Set `OPENAI_API_KEY` in your environment or replace the placeholder.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model="text-embedding-3-small",
)

## 4. Create and Run the Evaluation

The `AgentEvaluation` class is instantiated with the dataset, LLM config, and agent metrics. The `goal_achievement` and `response_coherence` metrics are built-in LLM-judge metrics.

In [ ]:
evaluation = AgentEvaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence"],
    default_provider="builtin",
)

results = evaluation.run()
print("Summary:", results.summary)

## 5. Inspect Per-Sample Results

Each sample result includes `user_input`, `final_response`, `reference_outcome`, and `metrics`. This enables per-sample analysis of evaluation quality.

In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr.get('user_input')}")
    print(f"  Final response: {sr.get('final_response', '')[:80]}...")
    for k, v in sr.get("metrics", {}).items():
        print(f"  {k}: score={v.get('score')}")

## Summary

This notebook demonstrated the evaluation of agents using pre-captured conversation traces.

The key components included:

1. **Agent Sample Structure**: An `AgentSample` with a complete trace (human, AI, tool, AI messages) was built and wrapped in an `AgentDataset`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for LLM-judge metrics.
3. **Evaluation Execution**: The `goal_achievement` and `response_coherence` metrics were run via the builtin provider.
4. **Results Inspection**: The summary and per-sample metrics were accessed through the results object.

This example showcases the workflow for evaluating agents when pre-recorded traces are available.